# Data: Rraw emails to eval split


In [8]:
%pip install -q polars==1.44.1 "pandera[polars]==0.33.1" scikit-learn==1.6.1

from pathlib import Path
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "demos" / "2.1-data").is_dir())
DEMO = ROOT / "demos" / "2.1-data"
os.chdir(DEMO)
sys.path.insert(0, str(ROOT / "demos"))
sys.path.insert(0, str(DEMO))

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import random
import re
from collections import Counter
from datetime import date
import hashlib
import json
from urllib.request import urlopen
from itertools import combinations
from pathlib import Path

import polars as pl

pl.Config(fmt_str_lengths=60, tbl_rows=8)

RAW = Path("data/raw")
PROC = Path("data/processed")
PROC.mkdir(parents=True, exist_ok=True)

CSV = RAW / "email_spam.csv"
RAW.mkdir(parents=True, exist_ok=True)
REVISION = "f9c3f31e0ab812ee86f17d9e497c3a35bcb7c200"
SOURCE = f"https://huggingface.co/datasets/UniqueData/email-spam-classification/resolve/{REVISION}/email_spam.csv"
if not CSV.exists():
    with urlopen(SOURCE, timeout=60) as response:
        payload = response.read()
    assert hashlib.md5(payload).hexdigest() == "2cd45a0a4c5ac2b8e5206c09b2355875"
    CSV.write_bytes(payload)
    (RAW / "ingest.json").write_text(json.dumps({"source": SOURCE, "retrieved_at": date.today().isoformat()}))
assert hashlib.md5(CSV.read_bytes()).hexdigest() == "2cd45a0a4c5ac2b8e5206c09b2355875", "Raw snapshot differs from the source pointer."
# Existing snapshots have an unknown retrieval date; do not invent one on rebuild.

raw = pl.read_csv(CSV)
print(raw.shape, raw.columns)

(84, 3) ['title', 'text', 'type']


## 1. Keep raw data immutable

DVC pointers will record the source repository, revision, and file hash.

In [10]:
print((RAW / "email_spam.csv.dvc").read_text())

md5: 4e4d91818cf096f9949c9ed2196f9da1
frozen: true
deps:
- path: email_spam.csv
  repo:
    url: https://huggingface.co/datasets/UniqueData/email-spam-classification
    rev_lock: f9c3f31e0ab812ee86f17d9e497c3a35bcb7c200
outs:
- md5: 2cd45a0a4c5ac2b8e5206c09b2355875
  size: 75702
  hash: md5
  path: email_spam.csv



## 2. Look at it

Read a few subjects and bodies. What does one row represent? Are the labels sensible?
The dataset's source and licence should be reviewed before reuse or redistribution.

In [11]:
raw.head(3)

title,text,type
str,str,str
"""?? the secrets to SUCCESS""","""Hi James, Have you claim your complimentary gift yet? I've…","""spam"""
"""?? You Earned 500 GCLoot Points""",""" alt_text Congratulations, you just earned 500 You complet…","""not spam"""
"""?? Your GitHub launch code""","""Here's your GitHub launch code, @Mortyj420! an octocat sta…","""not spam"""


## 3. Tidy

One row is one email. Rename `type` to `label`, map `not spam` to `ham`, and
strip surrounding whitespace. Hash the original title, text and label for a source ID.
Exact duplicate records share an ID; conflicting labels get different IDs. Sorting
by that ID makes downstream grouping independent of CSV row order.

`processed_at` records when we run this notebook, not when the source was collected.

In [12]:
tidy = (
    raw.rename({"type": "label"})
    .with_columns(pl.Series("email_id", [
        "w2-" + hashlib.sha256(json.dumps(row, ensure_ascii=False).encode()).hexdigest()[:16]
        for row in raw.rows()
    ]))
    .with_columns(
        pl.col("label").replace({"not spam": "ham"}),
        pl.col("title").str.strip_chars(),
        pl.col("text").str.strip_chars(),
        pl.lit(date.today()).alias("processed_at"),
    )
    .select("email_id", "title", "text", "label", "processed_at")
    .sort("email_id")
)

print(tidy.select("email_id", "title", "label").head(5))
print(tidy["label"].value_counts())

shape: (5, 3)
┌─────────────────────┬───────────────────────────────────────────────────────────────┬───────┐
│ email_id            ┆ title                                                         ┆ label │
│ ---                 ┆ ---                                                           ┆ ---   │
│ str                 ┆ str                                                           ┆ str   │
╞═════════════════════╪═══════════════════════════════════════════════════════════════╪═══════╡
│ w2-0295d0fa76e4c9aa ┆ Style with Denim                                              ┆ spam  │
│ w2-02cfd0ecd82617f4 ┆ Affordable American MBA degree ($180month)                    ┆ spam  │
│ w2-07370940782ceb15 ┆ 10-1 MLB Expert Inside, Plus Everything You Need To Have A B… ┆ spam  │
│ w2-0877fd93a3336c8a ┆ If coding in C++ is so extremely hard, why do we have so man… ┆ ham   │
│ w2-0e1f34dea0757a53 ┆ Venmo primary email address changed                           ┆ ham   │
└─────────────────────┴───

## 4. Cleaning

### 4a. Structural validity

Separate rows with missing or blank text, or unknown labels, before validation.
Then check the schema of the usable rows. A valid schema does not guarantee a
correct label.

In [13]:
import pandera.polars as pa

class Email(pa.DataFrameModel):
    email_id: str = pa.Field(str_matches=r"^w2-[0-9a-f]{16}$")
    title: str = pa.Field(str_length={"min_value": 1})
    text: str = pa.Field(str_length={"min_value": 1})
    label: str = pa.Field(isin=["spam", "ham"])

valid = (
    pl.col("title").is_not_null() & (pl.col("title").str.len_chars() > 0)
    & pl.col("text").is_not_null() & (pl.col("text").str.len_chars() > 0)
    & pl.col("label").is_in(["spam", "ham"])
).fill_null(False)
rejected = tidy.filter(~valid)
filtered = tidy.filter(valid)

Email.validate(filtered)
print(f"Schema holds: {filtered.height} rows; set aside: {rejected.height}")

Schema holds: 84 rows; set aside: 0


### 4b. Deduplication

In [14]:
dupes = (
    filtered.group_by("title", "text")
    .agg(
        pl.col("email_id"),
        pl.col("label").unique().sort().alias("labels"),
        pl.len().alias("n"),
    )
    .filter(pl.col("n") > 1)
)
print(dupes.select("email_id", "labels", "n"))

shape: (2, 3)
┌────────────────────────────────────────────────┬─────────────────┬─────┐
│ email_id                                       ┆ labels          ┆ n   │
│ ---                                            ┆ ---             ┆ --- │
│ list[str]                                      ┆ list[str]       ┆ u32 │
╞════════════════════════════════════════════════╪═════════════════╪═════╡
│ ["w2-394e871e17affcfe", "w2-39e495c4fc405ee9"] ┆ ["ham", "spam"] ┆ 2   │
│ ["w2-96e01d409d6ea289", "w2-96e01d409d6ea289"] ┆ ["ham"]         ┆ 2   │
└────────────────────────────────────────────────┴─────────────────┴─────┘


Treat duplicate groups according to their labels:

- Same email and same label: keep the first copy.
- Same email with conflicting labels: set aside every copy for review.

Choosing one conflicting label at random would hide a data quality problem.

In [ ]:
conflict = []
for row in dupes.filter(pl.col("labels").list.len() > 1).iter_rows(named=True):
    conflict.extend(row["email_id"])
print("quarantined for label conflict:", conflict)

deduped = (
    filtered.filter(~pl.col("email_id").is_in(conflict))
    .unique(subset=["title", "text"], keep="first", maintain_order=True)
)
print(f"rows {filtered.height} -> {deduped.height}")

### 4c. Personal information discussion

What personal information might remain in these emails? Structural validation
and deduplication do not remove it. Consider redaction and access controls in the lab.

## 5. Splitting: keep related emails together

In [ ]:
def shingles(s, k=5):
    toks = re.findall(r"[a-z0-9]+", s.lower())
    return {" ".join(toks[i : i + k]) for i in range(max(len(toks) - k + 1, 0))}

docs = [shingles(t + " " + b) for t, b in zip(deduped["title"], deduped["text"])]

parent = list(range(len(docs)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

for i, j in combinations(range(len(docs)), 2):
    union = docs[i] | docs[j]
    if union and len(docs[i] & docs[j]) / len(union) >= 0.5:
        parent[find(i)] = find(j)

campaign = [find(i) for i in range(len(docs))]
deduped = deduped.with_columns(pl.Series("campaign", campaign))

sizes = Counter(campaign)
for c, n in sizes.items():
    if n > 1:
        print(f"campaign of {n}:")
        for t in deduped.filter(pl.col("campaign") == c)["title"]:
            print("   ", t[:70])